# Lab Tuần 4 — Xử lý dữ liệu theo lô với PySpark


Yêu cầu trước khi bắt đầu: đã cài `pyspark` và JDK 11/17, và thư mục `data/`
nằm cùng cấp với thư mục chứa notebook này.

```bash
pip install pyspark
```
---

## Bài 0 — Khởi động

Mục tiêu: dựng `SparkSession`, kiểm tra môi trường và mở được Spark UI.

Chạy ô dưới. Nếu báo lỗi `JAVA_HOME`, cài JDK 11 hoặc 17 rồi chạy lại.
Ghi lại địa chỉ Spark UI — cả buổi lab sẽ dùng nó để quan sát Stage và Shuffle.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
import json, time

spark = (SparkSession.builder
         .appName("Lab-Tuan4")
         .master("local[*]")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

DATA = "../data"          # sua lai neu thu muc data nam cho khac
print("Spark", spark.version, "| so core:", sc.defaultParallelism)
print("Spark UI:", sc.uiWebUrl)

### Câu hỏi

Cụm đang chạy ở chế độ Cluster Manager nào? `local[*]` tương ứng bao nhiêu Executor?

*(Trả lời vào ô markdown bên dưới)*

_Trả lời:_ 

## Bài 1 — WordCount và vai trò của Combiner

Dữ liệu: `document.txt` (300 dòng văn bản).

Viết WordCount bằng **hai cách** rồi so sánh chi phí:
- **1a** dùng `reduceByKey`
- **1b** dùng `groupByKey` + `mapValues`

Sau khi chạy, mở tab **Stages** trên Spark UI, tìm cột *Shuffle Read / Shuffle Write*
của hai job và ghi lại con số.

In [ ]:
lines = sc.textFile(f"{DATA}/document.txt")

# TODO 1a: tach tu -> chuyen chu thuong -> dem bang reduceByKey
wc_reduce = None

# TODO 1b: lam lai bang groupByKey + mapValues
wc_group = None

print("Top 5:", sorted(wc_reduce.collect(), key=lambda x: -x[1])[:5])
print("Hai cach cho cung ket qua:",
      sorted(wc_reduce.collect()) == sorted(wc_group.collect()))

### Câu hỏi

Hai cách cho cùng kết quả. Vậy khác nhau ở đâu, và vì sao `reduceByKey` lại rẻ hơn?

*(Trả lời vào ô markdown bên dưới)*

_Trả lời:_ 

## Bài 2 — phép trung bình 

Dữ liệu: `temperatures.csv` — cột `station_id, year, temperature`.

Số bản ghi giữa các trạm **lệch rất mạnh**: TP.HCM có hơn 4000 bản ghi,
đỉnh Fansipan chỉ có 3. Đây là bài quan trọng nhất của buổi lab.

Tính nhiệt độ trung bình toàn quốc bằng **hai cách** rồi so sánh:
- **2a — cách sai**: tính trung bình của từng trạm trước, rồi lấy trung bình của các trung bình đó
- **2b — cách đúng**: dùng `aggregateByKey` giữ cặp `(tổng, số lượng)`, chỉ chia ở bước cuối

In [ ]:
raw = sc.textFile(f"{DATA}/temperatures.csv")
header = raw.first()
pairs = (raw.filter(lambda l: l != header)
            .map(lambda l: l.split(","))
            .map(lambda c: (c[0], float(c[2]))))      # (station_id, temperature)
print("So ban ghi:", pairs.count())

# TODO 2a: CACH SAI -- trung binh cua cac trung binh tung tram
tb_sai = None

# TODO 2b: CACH DUNG -- aggregateByKey giu (tong, so luong), gop toan bo roi moi chia
tb_dung = None

print("Trung binh SAI :", round(tb_sai, 2))
print("Trung binh DUNG:", round(tb_dung, 2))
print("Chenh lech     :", round(abs(tb_dung - tb_sai), 2), "do C")

> **Kết quả mong đợi:** Trung binh SAI ≈ 22.16 | DUNG ≈ 28.08 | chênh ≈ 5.92 °C

### Câu hỏi

Vì sao phép trung bình không dùng trực tiếp làm Combiner được, trong khi SUM, MAX, COUNT thì được?

*(Trả lời vào ô markdown bên dưới)*

_Trả lời:_ 

## Bài 3 — Accumulator và dữ liệu bẩn

Dữ liệu: `events.jsonl` — JSON Lines, trong đó có một số dòng **cố tình hỏng**.

Dùng `Accumulator` đếm số bản ghi không parse được **ngay trong lúc xử lý**,
không dùng thêm một Action riêng để đếm.

In [ ]:
loi_parse = sc.accumulator(0)

def parse_an_toan(dong):
    # TODO: tra ve dict neu parse duoc; neu loi thi cong accumulator va tra ve None
    pass

ban_ghi = sc.textFile(f"{DATA}/events.jsonl").map(parse_an_toan).filter(lambda x: x is not None)

print("So ban ghi hop le:", ban_ghi.count())
print("So ban ghi loi   :", loi_parse.value)

> **Kết quả mong đợi:** 2844 bản ghi hợp lệ, 156 bản ghi lỗi

### Câu hỏi

Nếu đọc `loi_parse.value` **trước** khi gọi `count()` thì được bao nhiêu? Và vì sao con số này có thể sai khi chạy trên cụm thật?

*(Trả lời vào ô markdown bên dưới)*

_Trả lời:_ 

## Bài 4 — DataFrame, Join và left_anti

Dữ liệu: `customers.csv` (600 khách) và `orders.csv`.

Tìm những khách hàng **chưa từng đặt đơn nào**, bằng hai cách,
rồi so sánh kế hoạch thực thi của chúng bằng `.explain()`.

In [ ]:
kh = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)
dh = spark.read.csv(f"{DATA}/orders.csv",    header=True, inferSchema=True)

# TODO 4a: dung left_anti
chua_mua_a = None

# TODO 4b: dung left join roi loc null
chua_mua_b = None

print("left_anti      :", chua_mua_a.count())
print("left join + null:", chua_mua_b.count())
chua_mua_a.explain()

> **Kết quả mong đợi:** Cả hai cách đều ra 151 khách hàng

### Câu hỏi

Hai cách cho cùng kết quả. Cách nào rẻ hơn và vì sao?

*(Trả lời vào ô markdown bên dưới)*

_Trả lời:_ 

## Bài 5 — Data skew và Custom Partitioner

Dữ liệu: `access.log` (định dạng `level ip method endpoint status latency`)
và `big_file.csv`.

**5a** Đếm số lượt truy cập theo `endpoint` và chỉ ra hiện tượng lệch tải.
**5b** Phân vùng `big_file.csv` theo domain của email bằng Custom Partitioner,
rồi đo **kích thước từng partition** để thấy phân bố không đều.

In [ ]:
# 5a -- do lech tai theo endpoint
logs = sc.textFile(f"{DATA}/access.log")
# TODO: tach cot thu 4 (index 3) la endpoint, dem theo endpoint, sap xep giam dan
phan_bo = None
for ep, n in phan_bo:
    print("%-16s %5d" % (ep, n))

# 5b -- custom partitioner theo domain email
def theo_domain(key):
    # TODO: tra ve so nguyen trong [0, 5) dua tren domain cua email
    pass

df = spark.read.csv(f"{DATA}/big_file.csv", header=True, inferSchema=True)
rdd = df.rdd.map(lambda r: (r["email"], r["amount"]))
# TODO: partitionBy voi 5 partition, roi dem so phan tu moi partition
kich_thuoc = None
print("Kich thuoc tung partition:", kich_thuoc)

> **Kết quả mong đợi:** `/api/search` chiếm ~55%, gấp ~25 lần endpoint nhỏ nhất

### Câu hỏi

Trong 5a, nếu mỗi endpoint đi về một Reducer riêng thì điều gì xảy ra với tổng thời gian chạy?

*(Trả lời vào ô markdown bên dưới)*

_Trả lời:_ 

## Bài 6 (nếu còn thời gian) — Đo tác động của cache()

Đo thời gian chạy `count()` hai lần trên cùng một RDD, khi **chưa** cache và khi **đã** cache.

In [ ]:
big = sc.textFile(f"{DATA}/big_file.csv").filter(lambda l: "gmail.com" in l)

# TODO: do thoi gian count() lan 1 va lan 2 khi CHUA cache
# TODO: sau do goi .cache(), chay count() mot lan de nap cache, roi do lai

### Câu hỏi

Khi chưa cache, vì sao lần `count()` thứ hai vẫn tốn gần bằng lần đầu?

*(Trả lời vào ô markdown bên dưới)*

_Trả lời:_ 

---
## Kết thúc

```python
spark.stop()
```